In [ ]:
# ============================================
# LIBRERÍAS REQUERIDAS
# ============================================
import pandas as pd
import matplotlib.pyplot as plt
import glob
import random

import csv  # Para lectura/escritura de archivos CSV línea por línea
import glob  # Para búsqueda de patrones de archivos
from datetime import datetime  # Para parseo de timestamps
from collections import defaultdict  # Para diccionarios con valores por defecto

In [ ]:
# ============================================
# PROCESAMIENTO DE DATOS DE COMPRAS
# ============================================
# Este script realiza dos pasadas sobre los archivos para:
# 1. Extraer todas las compras (eventos de purchase)
# 2. Calcular contexto de navegación previo (views y carts)

input_path = "./data/online+retail/split/*.csv"  # Ruta de archivos de entrada
output_file = "./data/purchases_processed.csv"  # Archivo de salida

print("\n=== INICIO DEL PROCESAMIENTO ===\n")

# ========== PASS 1: GUARDAR TODAS LAS COMPRAS ==========
print("PASS 1: Extrayendo eventos de compra...")
purchases = []

for file in glob.glob(input_path):
    with open(file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        
        for row in reader:
            # Filtrar solo eventos de tipo "purchase"
            if row["event_type"] == "purchase":
                purchases.append({
                    "event_time": row["event_time"],
                    "user_id": row["user_id"],
                    "product_id": row["product_id"],
                    "price": row["price"],
                    # Campos a ser completados en PASS 2
                    "view_count": 0,
                    "cart_count": 0,
                    "first_view": "",
                    "last_view": "",
                    "first_cart": "",
                    "last_cart": ""
                })

print(f"Se encontraron {len(purchases):,} compras")

# ========== PASS 2: CALCULAR ESTADÍSTICAS DE NAVEGACIÓN ==========
print("\nPASS 2: Calculando estadísticas de navegación previo a compra...")

# Diccionario para almacenar estadísticas por (usuario, producto)
stats = defaultdict(lambda: {
    "view_count": 0,  # Total de vistas
    "cart_count": 0,  # Total de intentos de carro
    "first_view": None,  # Timestamp primera vista
    "last_view": None,  # Timestamp última vista
    "first_cart": None,  # Timestamp primer carro
    "last_cart": None  # Timestamp último carro
})

# Recorrer todos los archivos nuevamente
for file in glob.glob(input_path):
    with open(file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = (row["user_id"], row["product_id"])  # Clave única usuario-producto
            event_time = datetime.strptime(row["event_time"], "%Y-%m-%d %H:%M:%S UTC")

            # ========== Procesar eventos VIEW ==========
            if row["event_type"] == "view":
                stats[key]["view_count"] += 1

                # Registrar primera vista (la más antigua)
                if stats[key]["first_view"] is None or event_time < stats[key]["first_view"]:
                    stats[key]["first_view"] = event_time

                # Registrar última vista (la más reciente)
                if stats[key]["last_view"] is None or event_time > stats[key]["last_view"]:
                    stats[key]["last_view"] = event_time

            # ========== Procesar eventos CART ==========
            if row["event_type"] == "cart":
                stats[key]["cart_count"] += 1

                # Registrar primer carro (el más antiguo)
                if stats[key]["first_cart"] is None or event_time < stats[key]["first_cart"]:
                    stats[key]["first_cart"] = event_time

                # Registrar último carro (el más reciente)
                if stats[key]["last_cart"] is None or event_time > stats[key]["last_cart"]:
                    stats[key]["last_cart"] = event_time

print(f"Se procesaron estadísticas para {len(stats):,} combinaciones usuario-producto")

# ========== COMBINAR ESTADÍSTICAS CON COMPRAS ==========
print("\nCombinando estadísticas con compras...")
for p in purchases:
    key = (p["user_id"], p["product_id"])

    # Si existe información de navegación para esta compra, enriquecerla
    if key in stats:
        s = stats[key]
        p["view_count"] = s["view_count"]
        p["cart_count"] = s["cart_count"]
        p["first_view"] = s["first_view"]
        p["last_view"] = s["last_view"]
        p["first_cart"] = s["first_cart"]
        p["last_cart"] = s["last_cart"]

print(f"Enriquecidas {len(purchases):,} compras con contexto de navegación")

# ========== GUARDAR RESULTADO EN CSV ==========
print(f"\nGuardando resultado en {output_file}...")
fieldnames = [
    "event_time", "user_id", "product_id", "price",
    "view_count", "cart_count",
    "first_view", "last_view",
    "first_cart", "last_cart"
]

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(purchases)

print(f"Archivo guardado exitosamente")
print(f"\n=== PROCESAMIENTO COMPLETADO ===\n")